in these days, I implemented both Integrated Gradients and SmoothGrad-IG, but ultimately abandoned them both. The final method used was Input × Gradient.

## **why?**

I started with attention heatmaps. to extract which nucleotide positions the model "looked at" most when making its decision. This worked technically, but a diagnostic revealed a fundamental problem: the model's attention was almost identical between pathogenic and benign sequences. The signal was dominated by structural artifacts (the [CLS] token attending to itself, boundary spikes), not genuine biological differences. So attention heatmaps were abandoned.


**The switch to Integrated Gradients:**

I then pivoted to Integrated Gradients (IG), which is a more principled method. It doesn't look at attention weights but instead it asks: "if I gradually increase this nucleotide's influence from zero, how much does the prediction change?"

I also implemented Signed IG (to show direction: does this position push toward pathogenic or benign?) and SmoothGrad-IG (to reduce noise by averaging over many runs with slight random perturbations).

**Why IG was also abandoned?**

DNABERT-2 uses a Triton-compiled "Flash Attention" kernel under the hood. Even after patching it to disable the forward path, the backward pass (which IG needs to accumulate gradients) still went through the Triton kernel and it produced unstable, non-converging results. The mathematical "completeness" check (which verifies the gradient integration is accurate) gave errors of δ > 0.4 across all configurations tested. That's far too high to trust.



**What I actually used: Input × Gradient**

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import os, warnings, types, math
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [ ]:
!pip install triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 8.4 MB/s eta 0:00:00


In [ ]:
mount = '/content/drive'
from google.colab import drive
drive.mount(mount)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── Reconstruct DNAClassifier ──────────────────────────────────────────────────
class DNAClassifier(nn.Module):
    def __init__(self, backbone, num_labels=2):
        super().__init__()
        self.backbone = backbone # Changed from self.bert to self.backbone
        hidden_size = backbone.config.hidden_size # Consistent with training definition
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), # Changed from hidden -> 256
            nn.Tanh(), # Changed from ReLU to Tanh
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels) # Changed from 256 -> num_labels
        )
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        # Handle both tuple and HuggingFace output object (hard-won fix from Day 7)
        if hasattr(outputs, "last_hidden_state"):
            hidden = outputs.last_hidden_state
        else:
            hidden = outputs[0]

        pooled = hidden[:, 0, :]         # [CLS] token = sequence summary
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

print("✓ DNAClassifier class defined")

# ── Load ORIGINAL Phase 2 model (F1=0.783) ────────────────────────────────────
# ── Updated paths for your actual folder structure ─────────────────────────────

# Where your fine-tuned weights and tokenizer files live
model_path = '/content/drive/MyDrive/project-finetuned'

# The architecture config comes from HuggingFace (no local file needed)
config = AutoConfig.from_pretrained('zhihan1996/DNABERT-2-117M', trust_remote_code=True)
if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = 0

# Build backbone from config
backbone = AutoModel.from_config(config, trust_remote_code=True)

# Wrap in classifier
model = DNAClassifier(backbone, num_labels=2)

# Load YOUR fine-tuned weights
model.load_state_dict(torch.load(
    os.path.join(model_path, 'dna_classifier.pt'),
    map_location=device
))

model = model.to(device)

# Load tokenizer from the same folder (your json files are there)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print(f'✓ Architecture loaded from HuggingFace')
print(f'✓ Fine-tuned weights loaded from {model_path}/dna_classifier.pt')
print(f'✓ Tokenizer loaded from {model_path}')

# Also update the data path
data_path = '/content/drive/MyDrive/project-finetuned/sequences_dataset.csv'
df = pd.read_csv(data_path)
print(f'✓ Dataset loaded: {df.shape}')
print(df.head(2))

# Apply Flash Attention patch — still needed to prevent the Triton crash
# Even though IG doesn't need attention weights, the forward pass still
# triggers Flash Attention unless we patch it
import glob, shutil
cache = os.path.expanduser('~/.cache/huggingface/modules/transformers_modules')
for f in glob.glob(f'{cache}/**/bert_layers.py', recursive=True):
    txt = open(f).read()
    if 'flash_attn_qkvpacked_func is not None' in txt:
        open(f, 'w').write(txt.replace(
            'if flash_attn_qkvpacked_func is not None',
            'if False  # patched'
        ))
shutil.rmtree(os.path.expanduser('~/.triton/cache'), ignore_errors=True)

# Apply monkey-patch to live model
def standard_attn_forward(self, hidden_states, cu_seqlens, max_seqlen_in_batch,
                           indices, attn_mask, bias):
    qkv         = self.Wqkv(hidden_states)
    hidden_size = qkv.shape[-1] // 3
    num_heads   = self.num_attention_heads
    head_dim    = hidden_size // num_heads
    batch_size  = cu_seqlens.shape[0] - 1
    max_s       = int(max_seqlen_in_batch)
    total_tokens = hidden_states.shape[0]

    q, k, v = qkv.split(hidden_size, dim=-1)

    if batch_size == 1:
        pad_len = max_s - total_tokens
        if pad_len > 0:
            zeros = torch.zeros(pad_len, hidden_size, dtype=q.dtype, device=q.device)
            q = torch.cat([q, zeros], dim=0)
            k = torch.cat([k, zeros], dim=0)
            v = torch.cat([v, zeros], dim=0)
        q = q.view(1, max_s, num_heads, head_dim).transpose(1, 2)
        k = k.view(1, max_s, num_heads, head_dim).transpose(1, 2)
        v = v.view(1, max_s, num_heads, head_dim).transpose(1, 2)
        mask = torch.zeros(1, 1, 1, max_s, dtype=torch.bool, device=q.device)
        if pad_len > 0:
            mask[0, 0, 0, total_tokens:] = True
    else:
        q_pad = torch.zeros(batch_size * max_s, hidden_size, dtype=q.dtype, device=q.device)
        k_pad = torch.zeros_like(q_pad)
        v_pad = torch.zeros_like(q_pad)
        q_pad = q_pad.index_put((indices,), q)
        k_pad = k_pad.index_put((indices,), k)
        v_pad = v_pad.index_put((indices,), v)
        q = q_pad.view(batch_size, max_s, num_heads, head_dim).transpose(1, 2)
        k = k_pad.view(batch_size, max_s, num_heads, head_dim).transpose(1, 2)
        v = v_pad.view(batch_size, max_s, num_heads, head_dim).transpose(1, 2)
        mask = torch.ones(batch_size, max_s, dtype=torch.bool, device=q.device)
        for b in range(batch_size):
            mask[b, :int(cu_seqlens[b+1] - cu_seqlens[b])] = False
        mask = mask[:, None, None, :]

    scores  = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
    scores  = scores.masked_fill(mask, -10000.0)
    weights = torch.softmax(scores.float(), dim=-1).to(q.dtype)
    weights = torch.nan_to_num(weights)
    context = torch.matmul(weights, v).transpose(1, 2).contiguous()
    context = context.view(-1, hidden_size)
    return context[:total_tokens] if batch_size == 1 else context[indices]

patched = 0
for name, module in model.backbone.named_modules():
    if type(module).__name__ == 'BertUnpadSelfAttention':
        module.forward = types.MethodType(standard_attn_forward, module)
        patched += 1

model.eval()
print(f'✓ Model loaded (original F1=0.783)')
print(f'✓ Flash Attention patched ({patched}/12 layers)')
print(f'✓ Tokenizer loaded')

✓ DNAClassifier class defined
✓ Architecture loaded from HuggingFace
✓ Fine-tuned weights loaded from /content/drive/MyDrive/project-finetuned/dna_classifier.pt
✓ Tokenizer loaded from /content/drive/MyDrive/project-finetuned
✓ Dataset loaded: (9995, 6)
                                     VariationID  \
0                      NM_000162.5(GCK):c.-84C>G   
1  NM_000038.6(APC):c.3189_3192del (p.Glu1064fs)   

                                            sequence  label  \
0  GGCACCCCTGGCAAGACCCTTCTCAAAGAGCCTGTGCATAAAATCA...      0   
1  TCAAATGAAACCCTCGATTGAATCCTATTCTGAAGATGATGAAAGT...      1   

  ClinicalSignificance Chromosome      Start  
0               Benign          7   44189037  
1           Pathogenic          5  112838781  
✓ Model loaded (original F1=0.783)
✓ Flash Attention patched (12/12 layers)
✓ Tokenizer loaded


In [ ]:
# ── Signed IG + SmoothGrad ────────────────────────────────────────────────────

def compute_ig_signed(sequence, model, tokenizer, device,
                      n_steps=100, target_class=None):
    """
    Signed Integrated Gradients.
    Returns positive scores (push toward target) AND negative scores
    (push away from target = toward the other class).

    The only difference from unsigned IG:
      unsigned: token_scores = ig_per_token.norm(dim=-1)   ← L2 magnitude
      signed:   token_scores = ig_per_token.sum(dim=-1)    ← signed sum
    """
    inputs    = tokenizer(sequence, return_tensors='pt', max_length=200,
                          truncation=True, padding='max_length',
                          add_special_tokens=True)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    token_ids = inputs['input_ids'][0].cpu().tolist()
    tokens    = tokenizer.convert_ids_to_tokens(token_ids)
    mask_np   = inputs['attention_mask'][0].cpu().numpy()
    n_real    = int(mask_np.sum())

    with torch.no_grad():
        real_embeddings = model.backbone.embeddings.word_embeddings(
            inputs['input_ids'])
        logits_check    = forward_from_embeddings(model, real_embeddings,
                                                   inputs['attention_mask'])
        probs           = torch.softmax(logits_check[0].cpu(), dim=0).numpy()
        predicted_class = int(np.argmax(probs))
        confidence      = float(probs[predicted_class])

    if target_class is None:
        target_class = predicted_class

    pad_id = tokenizer.pad_token_id or 0
    with torch.no_grad():
        baseline_embeddings = model.backbone.embeddings.word_embeddings(
            torch.full_like(inputs['input_ids'], pad_id))

    accumulated_grads = torch.zeros_like(real_embeddings)
    for step in range(n_steps):
        alpha  = step / (n_steps - 1)
        interp = (baseline_embeddings
                  + alpha * (real_embeddings - baseline_embeddings))
        interp = interp.detach().requires_grad_(True)
        logits = forward_from_embeddings(model, interp, inputs['attention_mask'])
        logits[0, target_class].backward()
        accumulated_grads += interp.grad.detach()

    avg_grads    = accumulated_grads / n_steps
    ig_per_token = (real_embeddings.detach()
                    - baseline_embeddings.detach()) * avg_grads
    # (1, seq_len, 768)

    # ── SIGNED: sum across embedding dimension ─────────────────────────────────
    # Positive sum = net push toward target class
    # Negative sum = net push away from target class
    token_scores_signed = ig_per_token[0].sum(dim=-1).cpu().numpy()  # (seq_len,)

    # Unsigned magnitude for comparison
    token_scores_unsigned = ig_per_token[0].norm(dim=-1).cpu().numpy()

    special = {'[CLS]','[SEP]','[PAD]','<cls>','<sep>','<pad>'}
    nuc_mask = np.array([t not in special for t in tokens[:n_real]])

    # Normalize signed scores to [-1, 1] — preserving zero as neutral
    signed = token_scores_signed[:n_real].copy()
    signed[~nuc_mask] = 0.0
    abs_max = np.abs(signed[nuc_mask]).max() + 1e-9
    signed_normalized = signed / abs_max   # [-1, 1]

    # Normalize unsigned to [0, 1] as before
    unsigned = token_scores_unsigned[:n_real].copy()
    unsigned[~nuc_mask] = 0.0
    u_min = unsigned[nuc_mask].min()
    u_max = unsigned[nuc_mask].max()
    unsigned_normalized = np.zeros(n_real)
    unsigned_normalized[nuc_mask] = (unsigned[nuc_mask] - u_min) / (u_max - u_min + 1e-9)

    # Delta check
    with torch.no_grad():
        baseline_logit = forward_from_embeddings(
            model, baseline_embeddings, inputs['attention_mask']
        )[0, target_class].item()
        real_logit = logits_check[0, target_class].item()
    delta = abs(ig_per_token[0].sum(dim=-1).sum().item()
                - (real_logit - baseline_logit))

    return {
        'signed':          signed_normalized,    # [-1, 1] — direction preserved
        'unsigned':        unsigned_normalized,  # [0, 1]  — magnitude only
        'tokens':          tokens[:n_real],
        'nuc_mask':        nuc_mask,
        'num_real_tokens': n_real,
        'predicted_class': predicted_class,
        'confidence':      confidence,
        'target_class':    target_class,
        'sequence':        sequence,
        'delta':           delta,
    }


def compute_smoothgrad_ig(sequence, model, tokenizer, device,
                           n_steps=100, n_samples=20, noise_level=0.15,
                           target_class=None):
    """
    SmoothGrad-IG: run IG n_samples times with Gaussian noise, average results.

    noise_level: noise std as fraction of embedding std (0.15 = 15% noise)
    n_samples:   how many noisy runs to average (20 is standard)

    WHY this helps:
    Single-run IG can hit sharp local features of the loss surface that
    aren't representative of the overall attribution pattern. Adding noise
    and averaging smooths these out — the true signal persists across
    noisy runs, random artifacts cancel.

    Cost: n_samples × slower than standard IG.
    With n_steps=100, n_samples=20 → 2000 forward/backward passes.
    ~8-10 minutes per sequence on T4.
    Reduce n_samples=10 if too slow.
    """
    inputs    = tokenizer(sequence, return_tensors='pt', max_length=200,
                          truncation=True, padding='max_length',
                          add_special_tokens=True)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    tokens    = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].cpu().tolist())
    mask_np   = inputs['attention_mask'][0].cpu().numpy()
    n_real    = int(mask_np.sum())

    with torch.no_grad():
        real_embeddings = model.backbone.embeddings.word_embeddings(
            inputs['input_ids'])
        logits_check    = forward_from_embeddings(model, real_embeddings,
                                                   inputs['attention_mask'])
        probs           = torch.softmax(logits_check[0].cpu(), dim=0).numpy()
        predicted_class = int(np.argmax(probs))
        confidence      = float(probs[predicted_class])

    if target_class is None:
        target_class = predicted_class

    pad_id = tokenizer.pad_token_id or 0
    with torch.no_grad():
        baseline_embeddings = model.backbone.embeddings.word_embeddings(
            torch.full_like(inputs['input_ids'], pad_id))

    # Noise scale: fraction of embedding standard deviation
    emb_std     = real_embeddings.detach().std().item()
    noise_std   = noise_level * emb_std

    # Accumulate IG maps across n_samples noisy copies
    all_ig_maps = []

    for sample_i in range(n_samples):
        # Add Gaussian noise to the real embeddings
        # WHY noise on real embeddings, not on interpolation path?
        # We perturb the endpoint — the whole interpolation path shifts
        # slightly, sampling a different slice of the gradient landscape
        noise           = torch.randn_like(real_embeddings) * noise_std
        noisy_real      = real_embeddings.detach() + noise

        accumulated_grads = torch.zeros_like(real_embeddings)
        for step in range(n_steps):
            alpha  = step / (n_steps - 1)
            interp = baseline_embeddings + alpha * (noisy_real - baseline_embeddings)
            interp = interp.detach().requires_grad_(True)
            logits = forward_from_embeddings(model, interp, inputs['attention_mask'])
            logits[0, target_class].backward()
            accumulated_grads += interp.grad.detach()

        avg_grads = accumulated_grads / n_steps
        ig_map    = (noisy_real - baseline_embeddings.detach()) * avg_grads
        # Signed sum per token
        ig_scores = ig_map[0].sum(dim=-1).cpu().numpy()   # (seq_len,)
        all_ig_maps.append(ig_scores)

        if (sample_i + 1) % 5 == 0:
            print(f"    SmoothGrad: {sample_i+1}/{n_samples} samples", flush=True)

    # Average across all noisy samples
    smoothed = np.mean(all_ig_maps, axis=0)   # (seq_len,)

    special  = {'[CLS]','[SEP]','[PAD]','<cls>','<sep>','<pad>'}
    nuc_mask = np.array([t not in special for t in tokens[:n_real]])

    smoothed_nuc = smoothed[:n_real].copy()
    smoothed_nuc[~nuc_mask] = 0.0

    # Normalize to [-1, 1]
    abs_max    = np.abs(smoothed_nuc[nuc_mask]).max() + 1e-9
    normalized = smoothed_nuc / abs_max

    return {
        'signed':          normalized,       # [-1, 1], averaged over n_samples
        'tokens':          tokens[:n_real],
        'nuc_mask':        nuc_mask,
        'num_real_tokens': n_real,
        'predicted_class': predicted_class,
        'confidence':      confidence,
        'target_class':    target_class,
        'sequence':        sequence,
        'n_samples':       n_samples,
        'noise_level':     noise_level,
    }


print("✓ compute_ig_signed() defined")
print("✓ compute_smoothgrad_ig() defined")

✓ compute_ig_signed() defined
✓ compute_smoothgrad_ig() defined


In [ ]:
# ── Diverging colormap heatmap for signed scores ───────────────────────────────

def plot_signed_heatmap(sequence, signed_scores, result_dict,
                         title=None, save_path=None, chars_per_row=60):
    """
    Diverging heatmap: red = pushes toward predicted class,
                       blue = pushes away from predicted class,
                       white = neutral
    """
    # RdBu_r: blue=-1 (away), white=0 (neutral), red=+1 (toward)
    cmap     = plt.cm.RdBu_r
    n_chars  = len(sequence)
    n_rows   = math.ceil(n_chars / chars_per_row)
    fig_w    = min(22, chars_per_row * 0.195)
    fig_h    = n_rows * 0.6 + 2.2
    fig, ax  = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    pred_name = 'PATHOGENIC' if result_dict['predicted_class'] == 1 else 'BENIGN'
    conf      = result_dict['confidence']
    method    = result_dict.get('n_samples', None)
    method_str = (f'SmoothGrad-IG ({method} samples)'
                  if method else 'Signed IG')

    if title is None:
        title = (f'{method_str} Attribution Map\n'
                 f'Prediction: {pred_name} ({conf:.1%})  |  '
                 f'Red = pushes toward {pred_name}  |  '
                 f'Blue = pushes away')
    ax.set_title(title, fontsize=10, pad=14, fontweight='bold')

    char_w  = 1.0 / chars_per_row
    row_h   = 1.0 / (n_rows + 0.5)
    norm    = mcolors.Normalize(vmin=-1, vmax=1)

    for i, (base, score) in enumerate(zip(sequence, signed_scores)):
        row = i // chars_per_row
        col = i  % chars_per_row
        x   = col * char_w
        y   = 1.0 - (row + 1) * row_h

        color = cmap(norm(float(score)))
        rect  = plt.Rectangle(
            (x, y - row_h * 0.08), char_w * 0.94, row_h * 0.82,
            color=color, transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)

        luminance  = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
        text_color = 'black' if luminance > 0.40 else 'white'
        ax.text(x + char_w*0.47, y + row_h*0.30, base,
                ha='center', va='center', fontsize=8.0,
                fontfamily='monospace', fontweight='bold',
                color=text_color, transform=ax.transAxes)

        if i % 10 == 0:
            ax.text(x + char_w*0.47, y - row_h*0.08, str(i),
                    ha='center', va='top', fontsize=5.0,
                    color='#555555', transform=ax.transAxes)

    # Diverging colorbar
    sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal',
                        fraction=0.025, pad=0.01, aspect=50)
    cbar.set_label(
        f'← pushes toward BENIGN        neutral        pushes toward PATHOGENIC →',
        fontsize=8.5)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.set_ticklabels(['-1.0', '-0.5', '0\n(neutral)', '+0.5', '+1.0'])

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"Saved: {save_path}")
    plt.show()
    return fig


print("✓ plot_signed_heatmap() defined")

✓ plot_signed_heatmap() defined


In [ ]:
# ── Quick comparison on the APC example: unsigned vs signed vs smoothgrad ──────
# Only run on the reliable APC sequence (delta=0.013) while screening continues

print("Running all three methods on APC example sequence...")
print()

# 1. Signed IG (fast — same cost as regular IG)
print("1/3 Signed IG...")
result_signed = compute_ig_signed(example_seq, model, tokenizer, device,
                                   n_steps=100)
nuc_signed = tokens_to_nucleotide_scores(
    result_signed['tokens'][1:-1],
    result_signed['signed'][1:-1],
    example_seq
)
# Renormalize signed scores to [-1,1] after nucleotide mapping
abs_max    = np.abs(nuc_signed).max() + 1e-9
nuc_signed = nuc_signed / abs_max

plot_signed_heatmap(
    sequence      = example_seq,
    signed_scores = nuc_signed,
    result_dict   = result_signed,
    save_path     = '/content/day12_signed_ig.png'
)

# 2. SmoothGrad-IG (slower — 20 samples × 100 steps = 2000 passes)
print("\n2/3 SmoothGrad-IG (20 samples — ~5 mins)...")
result_smooth = compute_smoothgrad_ig(
    example_seq, model, tokenizer, device,
    n_steps=100, n_samples=20, noise_level=0.15
)
nuc_smooth = tokens_to_nucleotide_scores(
    result_smooth['tokens'][1:-1],
    result_smooth['signed'][1:-1],
    example_seq
)
abs_max    = np.abs(nuc_smooth).max() + 1e-9
nuc_smooth = nuc_smooth / abs_max

plot_signed_heatmap(
    sequence      = example_seq,
    signed_scores = nuc_smooth,
    result_dict   = result_smooth,
    title         = (f'SmoothGrad-IG (20 samples, σ=15%) | '
                     f'APC pathogenic variant\n'
                     f'Red = pushes toward PATHOGENIC | '
                     f'Blue = pushes toward BENIGN'),
    save_path     = '/content/day12_smoothgrad_ig.png'
)

# 3. Side-by-side comparison
print("\n3/3 Side-by-side comparison...")
fig, axes = plt.subplots(3, 1, figsize=(20, 14))

for ax, (scores, label, cmap_name) in zip(axes, [
    (nuc_scores,  'Unsigned IG (magnitude)',        'RdYlBu_r'),
    (nuc_signed,  'Signed IG (direction preserved)', 'RdBu_r'),
    (nuc_smooth,  'SmoothGrad-IG (20 samples)',      'RdBu_r'),
]):
    cmap = plt.get_cmap(cmap_name)
    if 'Signed' in label or 'Smooth' in label:
        norm = mcolors.Normalize(vmin=-1, vmax=1)
    else:
        norm = mcolors.Normalize(vmin=0, vmax=1)

    colors = [cmap(norm(s)) for s in scores]
    bars   = ax.bar(range(len(scores)), scores, color=colors,
                    width=1.0, linewidth=0)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlim(-1, len(scores)+1)
    ax.set_xlabel("Nucleotide position")
    ax.set_ylabel("Score")
    ax.spines[['top','right']].set_visible(False)

    # Mark top 3 peaks
    top3 = np.argsort(np.abs(scores))[-3:]
    for pos in top3:
        ax.axvline(pos, color='black', linewidth=0.8,
                   linestyle='--', alpha=0.5)
        ax.text(pos, ax.get_ylim()[1]*0.9, str(pos),
                fontsize=7, ha='center', color='black')

fig.suptitle('Attribution method comparison — APC pathogenic variant',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/day12_method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

Running all three methods on APC example sequence...

1/3 Signed IG...


NameError: name 'example_seq' is not defined

In [ ]:
import torch

# ── Day 12 Cell 2: The Integrated Gradients function ──────────────────────────
#
# WHY this function is structured the way it is:
#
# Problem: token IDs are integers — you cannot differentiate through them.
# Solution: bypass the embedding lookup and work directly with embedding vectors.
#
# DNABERT-2's forward() normally does:
#   token_ids → embedding_layer → transformer layers → logit
# We intercept after the embedding layer and inject our own embeddings,
# allowing gradients to flow all the way back to the input.

def forward_from_embeddings(model, embeddings, attention_mask):
    backbone   = model.backbone
    batch_size, seq_len, _ = embeddings.shape

    token_type_ids    = torch.zeros(batch_size, seq_len,
                                    dtype=torch.long, device=embeddings.device)
    token_type_embeds = backbone.embeddings.token_type_embeddings(token_type_ids)
    embed_out         = embeddings + token_type_embeds
    embed_out         = backbone.embeddings.LayerNorm(embed_out)
    embed_out         = backbone.embeddings.dropout(embed_out)

    encoder_out = backbone.encoder(embed_out, attention_mask=attention_mask)
    cls_hidden  = encoder_out[-1][0].unsqueeze(0)   # (1, 768)

    # classifier is a Sequential — call it directly, dropout is inside it
    logits = model.classifier(cls_hidden)           # (1, 2)
    return logits

print("✓ forward_from_embeddings() fixed")
print("  model.classifier is Sequential(Linear→Tanh→Dropout→Linear)")
print("  No separate dropout needed — it's built into the Sequential")
print()
print("Re-run Cell 3 now.")


def compute_integrated_gradients(sequence, model, tokenizer, device,
                                  max_length=200, n_steps=50, target_class=1):
    """
    Compute Integrated Gradients attribution scores for each token position.

    Parameters
    ----------
    sequence     : str   — raw DNA sequence
    model        : DNAClassifier — in eval() mode
    tokenizer    : AutoTokenizer
    device       : torch.device
    max_length   : int   — must match training (200)
    n_steps      : int   — interpolation steps (50 is standard; more = more accurate)
    target_class : int   — 1=pathogenic, 0=benign
                           We compute attribution w.r.t. the pathogenic logit
                           so high scores = "this position pushed toward pathogenic"

    Returns
    -------
    dict with keys:
        'attributions'    : np.ndarray (n_real_tokens,) — one score per token
        'tokens'          : list[str]  — token strings
        'token_ids'       : list[int]
        'attention_mask'  : np.ndarray
        'num_real_tokens' : int
        'predicted_class' : int
        'confidence'      : float
        'sequence'        : str
        'delta'           : float — convergence check (should be < 0.05)
    """
    # ── Step 1: Tokenize ───────────────────────────────────────────────────────
    inputs = tokenizer(sequence, return_tensors='pt', max_length=max_length,
                       truncation=True, padding='max_length',
                       add_special_tokens=True)
    inputs     = {k: v.to(device) for k, v in inputs.items()}
    token_ids  = inputs['input_ids'][0].cpu().tolist()
    tokens     = tokenizer.convert_ids_to_tokens(token_ids)
    mask_np    = inputs['attention_mask'][0].cpu().numpy()
    n_real     = int(mask_np.sum())

    # ── Step 2: Get real embeddings ────────────────────────────────────────────
    # These are the raw word embeddings BEFORE positional encoding.
    # Shape: (1, seq_len, 768)
    with torch.no_grad():
        real_embeddings = model.backbone.embeddings.word_embeddings(
            inputs['input_ids']
        )   # (1, 200, 768)

    # ── Step 3: Define baseline ────────────────────────────────────────────────
    # The baseline = embedding of the padding token [PAD] for every position.
    # WHY PAD token? It represents "no information" — the neutral reference point.
    # IG measures: how much does each position contribute BEYOND the baseline?
    pad_id         = tokenizer.pad_token_id or 0
    pad_token_ids  = torch.full_like(inputs['input_ids'], pad_id)
    with torch.no_grad():
        baseline_embeddings = model.backbone.embeddings.word_embeddings(
            pad_token_ids
        )   # (1, 200, 768) — all padding embeddings

    # ── Step 4: Interpolate and accumulate gradients ───────────────────────────
    # Create n_steps interpolated embeddings between baseline and real
    # alpha=0.0 → baseline, alpha=1.0 → real input
    accumulated_grads = torch.zeros_like(real_embeddings)

    for step in range(n_steps):
        alpha = step / (n_steps - 1)   # 0.0, 0.02, 0.04, ..., 1.0

        # Interpolated embedding = baseline + alpha * (real - baseline)
        interp = (baseline_embeddings
                  + alpha * (real_embeddings - baseline_embeddings))
        interp = interp.detach().requires_grad_(True)
        # WHY requires_grad_(True)? This tells PyTorch to track operations
        # on this tensor so we can call .backward() and get d(output)/d(interp)

        # Forward pass from these embeddings
        logits = forward_from_embeddings(model, interp, inputs['attention_mask'])

        # Compute gradient of target class logit w.r.t. interpolated embeddings
        target_logit = logits[0, target_class]
        target_logit.backward()   # backpropagate

        # Accumulate gradient
        accumulated_grads += interp.grad.detach()

    # ── Step 5: Final IG formula ───────────────────────────────────────────────
    # IG = (real - baseline) × (1/n_steps) × sum_of_gradients
    avg_grads    = accumulated_grads / n_steps
    ig_per_token = (real_embeddings.detach() - baseline_embeddings.detach()) * avg_grads
    # ig_per_token shape: (1, seq_len, 768)
    # Each token has a 768-dim attribution vector

    # ── Step 6: Collapse 768-dim vector → scalar per token ────────────────────
    token_scores = ig_per_token[0].norm(dim=-1).cpu().numpy()   # (seq_len,)

    # ── Step 7: Normalize to [0, 1] ───────────────────────────────────────────
    raw_scores = token_scores[:n_real]   # trim padding — renamed to avoid conflict
    raw_scores = raw_scores - raw_scores.min()
    raw_scores = raw_scores / (raw_scores.max() + 1e-9)


    # ── Step 8: Get prediction + convergence check ─────────────────────────────
 # ── Step 8: Get prediction + convergence check ─────────────────────────────
    with torch.no_grad():
        logits_final = forward_from_embeddings(model, real_embeddings,
                                               inputs['attention_mask'])
    probs           = torch.softmax(logits_final[0].cpu(), dim=0).numpy()
    predicted_class = int(np.argmax(probs))
    confidence      = float(probs[predicted_class])

    with torch.no_grad():
        baseline_logit = forward_from_embeddings(
            model, baseline_embeddings, inputs['attention_mask']
        )[0, target_class].item()
        real_logit = logits_final[0, target_class].item()
    ig_sum = ig_per_token[0].sum(dim=-1).sum().item()
    delta  = abs(ig_sum - (real_logit - baseline_logit))

    return {
        'attributions':    raw_scores,       # ← was 'scores'
        'tokens':          tokens[:n_real],
        'token_ids':       token_ids[:n_real],
        'attention_mask':  mask_np,
        'num_real_tokens': n_real,
        'predicted_class': predicted_class,
        'confidence':      confidence,
        'sequence':        sequence,
        'delta':           delta,
    }

print('✓ compute_integrated_gradients() defined')
print()
print('Key design decisions:')
print('  Baseline = PAD token embeddings (represents "no information")')
print('  n_steps  = 50 interpolation steps (standard; 20 is fast, 300 is precise)')
print('  Collapse = L2 norm of 768-dim attribution vector per token')
print('  Target   = class 1 (pathogenic) logit')

✓ forward_from_embeddings() fixed
  model.classifier is Sequential(Linear→Tanh→Dropout→Linear)
  No separate dropout needed — it's built into the Sequential

Re-run Cell 3 now.
✓ compute_integrated_gradients() defined

Key design decisions:
  Baseline = PAD token embeddings (represents "no information")
  n_steps  = 50 interpolation steps (standard; 20 is fast, 300 is precise)
  Collapse = L2 norm of 768-dim attribution vector per token
  Target   = class 1 (pathogenic) logit


In [ ]:
import pandas as pd

data_path = '/content/drive/MyDrive/project-finetuned/sequences_dataset.csv'
df        = pd.read_csv(data_path)
SEQ_COL, LABEL_COL = 'sequence', 'label'

In [ ]:
# ── Day 12 Cell 3: Test on one sequence + validate delta ──────────────────────

# Test on one pathogenic sequence
example_seq  = df[df[LABEL_COL] == 1][SEQ_COL].iloc[0]
print(f'Sequence length: {len(example_seq)} characters')
print(f'Running IG with 50 steps...')
print()

result = compute_integrated_gradients(example_seq, model, tokenizer, device)

print(f'✓ Done')
print(f'  Real tokens:     {result["num_real_tokens"]}')
print(f'  Attribution shape: {result["attributions"].shape}')
print(f'  Predicted:       {"PATHOGENIC" if result["predicted_class"]==1 else "BENIGN"}')
print(f'  Confidence:      {result["confidence"]:.1%}')
print(f'  Delta (convergence check): {result["delta"]:.4f}')
print(f'    → Should be < 0.05. If larger, increase n_steps.')
print()
print('Top 5 most attributed token positions:')
top5 = np.argsort(result['attributions'])[-5:][::-1]
for rank, pos in enumerate(top5):
    print(f'  #{rank+1}: position {pos:3d} | '
          f'token = {result["tokens"][pos]!r:12s} | '
          f'score = {result["attributions"][pos]:.4f}')

1- Delta = 1.18 (should be < 0.05). the gradients aren't converging. 50 steps is nowhere near enough.

2-[CLS] gets score 1.0. same boundary token dominance problem we saw with rollout. The gradient is flowing most strongly to the [CLS] token because that's literally where the classifier reads from.

In [ ]:
# ── Diagnostic: find the right n_steps and fix CLS dominance ──────────────────

# Check delta at increasing step counts to find convergence point
print("Delta convergence test (higher steps = more accurate):")
print(f"{'Steps':<8} {'Delta':>8} {'Converged?':>12}")
print("-" * 32)

for n_steps in [50, 100, 200, 300]:
    r = compute_integrated_gradients(
        example_seq, model, tokenizer, device,
        n_steps=n_steps
    )
    converged = "✓" if r['delta'] < 0.1 else "✗"
    print(f"  {n_steps:<6} {r['delta']:>8.4f} {converged:>12}")

# Fix

In [ ]:
# ── Cell 4: Map token scores → individual nucleotide letters ──────────────────

def tokens_to_nucleotide_scores(tokens, attributions, sequence):
    """
    Distribute token-level IG scores onto individual nucleotide positions.

    DNABERT-2 uses k-mer tokenization with a sliding window — tokens overlap.
    Token at position i covers nucleotides starting at i and extending for
    len(token) characters. Where tokens overlap, we average their scores.

    Example with 3-mers on "ATCGA":
      Token 0: "ATC" covers positions 0,1,2
      Token 1: "TCG" covers positions 1,2,3
      Token 2: "CGA" covers positions 2,3,4
      Position 2 (C) gets average of tokens 0,1,2
    """
    seq_len = len(sequence)
    scores  = np.zeros(seq_len)
    counts  = np.zeros(seq_len)

    nuc_pos = 0  # current position in the nucleotide sequence

    for tok_idx, token in enumerate(tokens):
        # Skip special tokens — they don't map to nucleotides
        if token in ('[CLS]', '[SEP]', '[PAD]', '<cls>', '<sep>', '<pad>'):
            continue

        # Clean tokenizer artifacts
        clean = token.replace('##', '').replace('▁', '').strip()
        if len(clean) == 0:
            continue

        # Distribute this token's attribution across the nucleotides it covers
        for offset in range(len(clean)):
            pos = nuc_pos + offset
            if pos < seq_len:
                scores[pos] += attributions[tok_idx]
                counts[pos] += 1

        nuc_pos += 1   # k-mer window slides one nucleotide at a time

    # Average overlapping contributions
    valid         = counts > 0
    scores[valid] = scores[valid] / counts[valid]

    # Normalize to [0, 1] over the whole sequence
    scores = scores - scores.min()
    scores = scores / (scores.max() + 1e-9)

    return scores


# Apply to our example
nuc_scores = tokens_to_nucleotide_scores(
    tokens       = result['tokens'][1:-1],      # strip [CLS] and [SEP]
    attributions = result['attributions'][1:-1], # matching scores
    sequence     = example_seq
)

print(f"Sequence length:          {len(example_seq)} nucleotides")
print(f"Nucleotide scores length: {len(nuc_scores)}")
print(f"Score range: [{nuc_scores.min():.4f}, {nuc_scores.max():.4f}]")
print()

# Find the highest-scoring nucleotide region
top10_nuc = np.argsort(nuc_scores)[-10:][::-1]
print("Top 10 individual nucleotide positions:")
for rank, pos in enumerate(top10_nuc):
    print(f"  #{rank+1:2d}: position {pos:4d} | "
          f"base={example_seq[pos]} | "
          f"score={nuc_scores[pos]:.4f}")
print()
print("Context around top position:")
top_pos = top10_nuc[0]
start   = max(0, top_pos - 10)
end     = min(len(example_seq), top_pos + 10)
print(f"  ...{example_seq[start:top_pos]}[{example_seq[top_pos]}]{example_seq[top_pos+1:end]}...")
print(f"  Position {top_pos} in the 513-char sequence")

In [ ]:
# ── Cell 5: Publication-style attribution heatmap ─────────────────────────────

def plot_attribution_heatmap(sequence, nuc_scores, result,
                              title=None, save_path=None,
                              chars_per_row=60):
    """
    Color each nucleotide letter by its IG attribution score.
    Blue = low attribution (unimportant for classification)
    Red  = high attribution (model relied on this position)
    """
    cmap      = plt.cm.RdYlBu_r   # blue→yellow→red
    n_chars   = len(sequence)
    n_rows    = math.ceil(n_chars / chars_per_row)

    fig_w  = min(22, chars_per_row * 0.195)
    fig_h  = n_rows * 0.6 + 2.0
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')

    pred_name = 'PATHOGENIC' if result['predicted_class'] == 1 else 'BENIGN'
    if title is None:
        title = (f'Integrated Gradients Attribution Map\n'
                 f'Prediction: {pred_name}  ({result["confidence"]:.1%} confidence) | '
                 f'delta={result["delta"]:.4f} | '
                 f'{n_chars} nucleotides')
    ax.set_title(title, fontsize=10, pad=14, fontweight='bold')

    char_w   = 1.0 / chars_per_row
    row_h    = 1.0 / (n_rows + 0.5)
    font_sz  = 8.0

    for i, (base, score) in enumerate(zip(sequence, nuc_scores)):
        row = i // chars_per_row
        col = i  % chars_per_row

        x = col * char_w
        y = 1.0 - (row + 1) * row_h

        color = cmap(float(score))

        # Background rectangle colored by attribution
        rect = plt.Rectangle(
            (x, y - row_h * 0.08),
            char_w * 0.94,
            row_h * 0.82,
            color=color,
            transform=ax.transAxes,
            clip_on=False
        )
        ax.add_patch(rect)

        # Nucleotide letter — black on light backgrounds, white on dark
        luminance  = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
        text_color = 'black' if luminance > 0.40 else 'white'

        ax.text(
            x + char_w * 0.47,
            y + row_h * 0.30,
            base,
            ha='center', va='center',
            fontsize=font_sz,
            fontfamily='monospace',
            fontweight='bold',
            color=text_color,
            transform=ax.transAxes
        )

        # Position label every 10 bases
        if i % 10 == 0:
            ax.text(
                x + char_w * 0.47,
                y - row_h * 0.08,
                str(i),
                ha='center', va='top',
                fontsize=5.0,
                color='#555555',
                transform=ax.transAxes
            )

    # Colorbar
    sm   = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal',
                        fraction=0.025, pad=0.01, aspect=50)
    cbar.set_label('IG Attribution Score   (blue = low importance,  red = high importance)',
                   fontsize=8.5)
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cbar.set_ticklabels(['0.0', '0.25', '0.5', '0.75', '1.0'])

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight',
                    facecolor='white', edgecolor='none')
        print(f"Saved: {save_path}")

    plt.show()
    return fig


# ── Generate heatmap for our example sequence ──────────────────────────────────
fig = plot_attribution_heatmap(
    sequence   = example_seq,
    nuc_scores = nuc_scores,
    result     = result,
    save_path  = '/content/day12_example_heatmap.png'
)

In [ ]:
# ── Biological context lookup before running Cell 6 ───────────────────────────
match = df[df[SEQ_COL] == example_seq]
print("Variant information:")
print(match[['VariationID', 'ClinicalSignificance', 'Chromosome', 'Start']].to_string())
print()

seq_len = len(example_seq)
center  = seq_len // 2
print(f"Sequence center (variant position): {center}")
print(f"Hotspot 1: position 48  → {center - 48} nt upstream of variant")
print(f"Hotspot 2: position 83-90 → {center - 86} nt upstream of variant")
print()

# Print the actual hotspot sequences
print(f"Hotspot 1 context:  ...{example_seq[43:53]}...")
print(f"                        ↑ pos 48")
print()
print(f"Hotspot 2 context:  ...{example_seq[78:95]}...")
print(f"                           ↑↑↑↑↑↑↑↑ pos 83-90")
print()

# Check for known motifs in hotspot regions
print("Checking hotspot sequences:")
h1 = example_seq[45:52]
h2 = example_seq[80:93]
print(f"  Hotspot 1 (±3):  {h1}")
print(f"  Hotspot 2 (±3):  {h2}")
print()
print("Things to look up for Day 13:")
print(f"  1. Is '{h1}' a known splice acceptor/donor motif? (GT...AG rule)")
print(f"  2. Is '{h2}' a TATA box or transcription factor binding site?")
print(f"     TATA box consensus: TATAAA — your hotspot 2 contains 'ATAAA'")
print(f"  3. Search both sequences in JASPAR or UniPROBE for TF binding")

In [ ]:
# ── Day 12 Cell 6: Run on 10 sequences — 5 pathogenic, 5 benign ───────────────
# This is the main scientific test: do pathogenic and benign sequences
# show visually different attribution patterns?

output_dir = '/content/drive/MyDrive/project-finetuned/heatmaps'
os.makedirs(output_dir, exist_ok=True)

print('Generating heatmaps for 10 sequences...')
print('(~30 seconds per sequence with n_steps=50)\n')

summary = []

for label in [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]:
    class_name = 'pathogenic' if label == 1 else 'benign'
    seq        = df[df[LABEL_COL] == label][SEQ_COL].sample(1).iloc[0]

    print(f'Processing {class_name}...', end=' ', flush=True)
    result     = compute_integrated_gradients(seq, model, tokenizer, device)
    nuc_scores = tokens_to_nucleotide_scores(
        result['tokens'][1:-1],
        result['attributions'][1:-1],
        seq
    )

    pred_name  = 'PATHOGENIC' if result['predicted_class'] == 1 else 'BENIGN'
    correct    = '✓' if result['predicted_class'] == label else '✗'
    print(f'{pred_name} ({result["confidence"]:.0%}) {correct} | '
          f'delta={result["delta"]:.3f}')

    # Save individual heatmap
    fname = f'{class_name}_{len(summary)+1:02d}.png'
    plot_attribution_heatmap(
        sequence   = seq,
        nuc_scores = nuc_scores,
        result     = result,
        title      = (f'True: {class_name.upper()} | '
                      f'Pred: {pred_name} ({result["confidence"]:.0%})\n'
                      f'Integrated Gradients Attribution | delta={result["delta"]:.3f}'),
        save_path  = os.path.join(output_dir, fname)
    )
    plt.close()

    summary.append({
        'true_label':  class_name,
        'predicted':   pred_name.lower(),
        'correct':     result['predicted_class'] == label,
        'confidence':  result['confidence'],
        'delta':       result['delta'],
        'top_pos':     int(np.argmax(nuc_scores)),
        'top_base':    seq[int(np.argmax(nuc_scores))],
        'entropy':     float(-np.sum((nuc_scores/nuc_scores.sum()+1e-9) *
                             np.log(nuc_scores/nuc_scores.sum()+1e-9))),
    })

print()
print('═' * 60)
print('SUMMARY')
print('═' * 60)
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print()

path_entropy = summary_df[summary_df.true_label=='pathogenic']['entropy'].mean()
ben_entropy  = summary_df[summary_df.true_label=='benign']['entropy'].mean()
print(f'Mean attribution entropy:')
print(f'  Pathogenic: {path_entropy:.3f}')
print(f'  Benign:     {ben_entropy:.3f}')
print(f'  Difference: {ben_entropy - path_entropy:+.3f}')
print()
print(f'Positive = pathogenic more focused (expected if model learned variant sites)')
print(f'Saved all heatmaps to: {output_dir}')

In [ ]:
# ── Find sequences where the model is actually confident ──────────────────────
# Screen a larger set and keep only the ones with good delta and confidence

print("Screening sequences for reliable IG attribution...")
print("Criteria: confidence > 0.65 AND delta < 0.1\n")

reliable = []
screened = 0

for label in [1, 0]:
    class_name = 'pathogenic' if label == 1 else 'benign'
    candidates = df[df[LABEL_COL] == label][SEQ_COL].tolist()

    for seq in candidates[:50]:   # check up to 50 per class
        screened += 1

        # Quick confidence check first (fast — no IG needed)
        inp = tokenizer(seq, return_tensors='pt', max_length=200,
                       truncation=True, padding='max_length')
        inp = {k: v.to(device) for k, v in inp.items()}

        with torch.no_grad():
            emb    = model.backbone.embeddings.word_embeddings(inp['input_ids'])
            logits = forward_from_embeddings(model, emb, inp['attention_mask'])
            probs  = torch.softmax(logits[0].cpu(), dim=0).numpy()
            conf   = float(probs.max())
            pred   = int(np.argmax(probs))

        # Only run expensive IG if model is confident
        if conf < 0.6:
            continue

        # Run IG with higher steps for accuracy
        res = compute_integrated_gradients(seq, model, tokenizer, device,
                                           n_steps=100)

        if res['delta'] < 0.1:
            nuc_sc = tokens_to_nucleotide_scores(
                res['tokens'][1:-1],
                res['attributions'][1:-1],
                seq
            )
            reliable.append({
                'sequence':   seq,
                'label':      label,
                'class_name': class_name,
                'result':     res,
                'nuc_scores': nuc_sc,
                'confidence': conf,
                'delta':      res['delta'],
                'correct':    pred == label,
            })
            status = '✓' if pred == label else '✗'
            print(f"  KEPT [{len(reliable):2d}] {class_name:10s} | "
                  f"conf={conf:.1%} | delta={res['delta']:.4f} {status}")

        if len(reliable) >= 10:   # stop when we have enough
            break

    if len(reliable) >= 10:
        break

print(f"\nScreened {screened} sequences → {len(reliable)} reliable")

# Debugging

In [ ]:
# ── Inspect encoder output shape ───────────────────────────────────────────────
with torch.no_grad():
    inputs = tokenizer("ATCGATCG", return_tensors='pt',
                       max_length=200, truncation=True, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get word embeddings
    word_embeds = model.backbone.embeddings.word_embeddings(inputs['input_ids'])
    token_type_ids = torch.zeros_like(inputs['input_ids'])
    token_type_embeds = model.backbone.embeddings.token_type_embeddings(token_type_ids)
    embed_out = word_embeds + token_type_embeds
    embed_out = model.backbone.embeddings.LayerNorm(embed_out)
    embed_out = model.backbone.embeddings.dropout(embed_out)

    print(f"embed_out shape: {embed_out.shape}")

    encoder_out = model.backbone.encoder(embed_out, attention_mask=inputs['attention_mask'])

    print(f"encoder_out type: {type(encoder_out)}")
    if hasattr(encoder_out, 'last_hidden_state'):
        print(f"last_hidden_state shape: {encoder_out.last_hidden_state.shape}")
    elif isinstance(encoder_out, (tuple, list)):
        print(f"tuple/list of length: {len(encoder_out)}")
        for i, x in enumerate(encoder_out):
            if hasattr(x, 'shape'):
                print(f"  [{i}] shape: {x.shape}")
    else:
        print(f"raw shape: {encoder_out.shape}")

In [ ]:
# ── Inspect: which of the 12 tensors is the final layer, and where is CLS? ────
with torch.no_grad():
    inputs = tokenizer("ATCGATCG", return_tensors='pt',
                       max_length=200, truncation=True, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Also run the full backbone normally to compare
    full_out = model.backbone(input_ids=inputs['input_ids'],
                              attention_mask=inputs['attention_mask'])

    print(f"Full backbone output type: {type(full_out)}")
    if hasattr(full_out, 'last_hidden_state'):
        print(f"last_hidden_state shape: {full_out.last_hidden_state.shape}")
        print(f"last_hidden_state[:, 0, :5]: {full_out.last_hidden_state[0, 0, :5]}")

    # Now check what the encoder list contains
    word_embeds = model.backbone.embeddings.word_embeddings(inputs['input_ids'])
    token_type_ids = torch.zeros_like(inputs['input_ids'])
    token_type_embeds = model.backbone.embeddings.token_type_embeddings(token_type_ids)
    embed_out = word_embeds + token_type_embeds
    embed_out = model.backbone.embeddings.LayerNorm(embed_out)
    embed_out = model.backbone.embeddings.dropout(embed_out)

    encoder_out = model.backbone.encoder(embed_out, attention_mask=inputs['attention_mask'])

    print(f"\nEncoder list[-1] shape: {encoder_out[-1].shape}")
    print(f"Encoder list[-1][0, :5]: {encoder_out[-1][0, :5]}")
    print()

    # Check if the full backbone also stores indices somewhere
    print("Full backbone output attributes:")
    for attr in dir(full_out):
        if not attr.startswith('_'):
            val = getattr(full_out, attr, None)
            if val is not None and not callable(val):
                print(f"  {attr}: {type(val)}")

In [ ]:
# ── See exactly what DNAClassifier contains this session ──────────────────────
print("model.named_children():")
for name, module in model.named_children():
    print(f"  '{name}': {type(module).__name__} — {module}")
print()
print("model.__dict__ keys:", [k for k in model.__dict__ if not k.startswith('_')])